# CPCB air quality - pipeline walkthrough
HDFS storage, Bronze/Silver/Gold layers, CPCB AQI, Spark SQL, RDDs and Hive.

In [ ]:
import sys, subprocess; sys.path.insert(0, '..')
from common import *
from pyspark.sql import functions as F
spark = get_spark('walkthrough', hive=True)
spark.version

## 1. Data lake in HDFS

In [ ]:
print(subprocess.run(['bash', '-c', 'source ../hadoop/hadoop_env.sh && hdfs dfs -du -h /cpcb/lake'],
                     capture_output=True, text=True).stdout)

## 2. Silver layer: cleaned, enriched, partitioned by year

In [ ]:
silver = spark.read.parquet(SILVER_DIR)
print(f'{silver.count():,} clean rows')
silver.select('station_id', 'state', 'city', 'ts', 'season', 'hour', 'pm25').show(5)

In [ ]:
import pandas as pd
pd.read_csv(TABLES_DIR / 'cleaning_log.csv')[['rule', 'rows_after', 'rows_removed', 'values_nulled']]

## 3. Gold layer: daily CPCB AQI

In [ ]:
daily = spark.read.parquet(GOLD_DAILY_DIR)
daily.createOrReplaceTempView('daily')
daily.filter("city = 'Delhi' AND date = '2021-11-05'").select('station_id', 'pm25', 'pm10', 'co8', 'o38', 'aqi', 'aqi_category', 'prominent_pollutant').show(5)

## 4. Spark SQL: most polluted cities (RQ1)

In [ ]:
spark.sql('''
SELECT city, state, ROUND(AVG(pm25), 1) AS mean_pm25, ROUND(AVG(aqi)) AS mean_aqi,
       ROUND(100 * AVG(CASE WHEN aqi > 200 THEN 1.0 ELSE 0 END), 1) AS pct_poor
FROM daily WHERE year BETWEEN 2019 AND 2022 AND aqi IS NOT NULL
GROUP BY city, state HAVING COUNT(DISTINCT date) >= 600 AND COUNT(pm25) >= 300
ORDER BY mean_pm25 DESC LIMIT 10''').show()

## 5. RDD transformations on the raw HDFS files

In [ ]:
sc = spark.sparkContext
lines = sc.textFile(f'{RAW_URI}/DL001.csv')
header = lines.first()
pm_idx = header.split(',').index('PM2.5 (ug/m3)')
by_year = (lines.filter(lambda l: l != header)
                .map(lambda l: l.split(','))
                .filter(lambda f: f[pm_idx] not in ('', None))
                .map(lambda f: (f[0][:4], (float(f[pm_idx]), 1)))
                .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
                .mapValues(lambda v: round(v[0] / v[1], 1))
                .sortByKey())
print(by_year.collect())
print(by_year.toDebugString().decode()[:600])

In [ ]:
pd.read_csv(TABLES_DIR / 'rdd_validation.csv')

## 6. Hive tables

In [ ]:
spark.sql('SHOW TABLES IN cpcb').show()

In [ ]:
spark.sql('DESCRIBE FORMATTED cpcb.daily_aqi').filter("col_name IN ('Type','Location','Serde Library','InputFormat')").show(truncate=False)

In [ ]:
spark.sql('''
SELECT state, city, mean_pm25 FROM cpcb.city_year_orc
WHERE year = 2022 AND days >= 200 ORDER BY mean_pm25 DESC LIMIT 8''').show()

## 7. Partition pruning in the physical plan

In [ ]:
spark.sql("SELECT state, AVG(pm25) FROM daily WHERE year = 2022 GROUP BY state").explain()

In [ ]:
spark.stop()